In [1]:
import google.auth
import numpy as np
import pandas as pd
import pygris 
import pyogrio
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase
from shared_utils import arcgis_query

In [2]:
from calitp_data_analysis import get_fs
fs = get_fs()

In [3]:

import os
from typing import List, Optional, Union
import pyarrow.dataset as ds
from google.cloud import storage

In [4]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
from calitp_data_analysis.sql import to_snakecase

In [84]:
from typing import List

In [5]:
import geopandas as gpd
import pandas as pd

In [6]:
from functools import cache

from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [12]:
gcsgp = GCSGeoPandas()

# Census Blocks

In [7]:
census_year = 2020

In [8]:
census_gdf = to_snakecase(gcs_geopandas().read_parquet(f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_combined_{census_year}.parquet"))

In [9]:
census_gdf.shape

(519723, 19)

In [26]:
census_gdf.crs

<Projected CRS: ESRI:102600>
Name: NAD_1983_California_Teale_Albers_FtUS
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: NAD_1983_California_Teale_Albers_FtUS
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

## HPMS

In [73]:
def load_hpms(url:str)->gpd.GeoDataFrame:
    df = to_snakecase(gcsgp.read_parquet(
    hpms_url)).to_crs(geography_utils.CA_NAD83Albers_ft)
    return df

In [10]:
hpms_url = "gs://calitp-analytics-data/data-analyses/equity_index/HPMS21_Main_SUCU.parquet"

In [74]:
hpm_df = load_hpms(hpms_url)

In [30]:
hpm_df.head(2)

,orderid,routeid,beginpoint,endpoint,sectionlength,lanemile,f_system,nhs,urbanid,facility_type,ownership,county_id,through_lanes,aadt,dvmt_1000,avmt_million,aadt_single_unit,aadt_combination,shape_length,geometry
0,1517,ALA_CO_CHAPARRAL LN_P,0.0,0.124,0.124,0.248,7,NaN,78904,2,2,1,2,450,0.056,0.020,NaN,NaN,199.031795,"MULTILINESTRING ((-591898.921 9012384.152, -59..."
1,1516,ALA_CO_CHANNEL ST_P,0.0,0.579,0.579,1.158,5,NaN,78904,2,2,1,2,10711,6.202,2.264,NaN,NaN,931.200265,"MULTILINESTRING ((-617685.094 9003972.798, -61..."


In [31]:
hpm_df.f_system.unique()

array([7, 5, 4, 3, 6, 2, 1], dtype=int32)

In [32]:
# Filter to f_system 1,2
hpm_df2 = hpm_df.loc[hpm_df.f_system.isin([1,2])]

## Main function

In [35]:
buffer_list = [500, 450, 400, 350, 300, 250, 200, 150, 100, 50]

In [83]:
len(buffer_list)

10

In [80]:
def buffer_intersect(hpm_gdf:gpd.GeoDataFrame, census_gdf:gpd.GeoDataFrame, buffer: int)->pd.DataFrame:
    hpm_gdf.geometry = hpm_gdf.geometry.buffer(buffer)
    hpm_gdf[f"{buffer}_area"] = hpm_gdf.geometry.area
    
    intersect = gpd.overlay(census_gdf, hpm_gdf)

    # Create an unique ID 
    intersect["unique_id"] = intersect.geoid20 + "_" + intersect.routeid

    # Find max value per unique ID
    intersect[f"aadt_{buffer}"] = intersect.groupby("unique_id")["aadt"].transform("max")

    # Sum the maximum AADT by GEOID
    agg = intersect.groupby("geoid20").agg({f"aadt_{buffer}":"sum"}).reset_index()

    return agg

In [81]:
buffer_500 = buffer_intersect(hpm_gdf=hpm_df2, census_gdf=census_gdf, buffer = buffer_list[0])

In [86]:

def run_buffers_and_merge(
    hpm_gdf: gpd.GeoDataFrame,
    census_gdf: gpd.GeoDataFrame,
    buffer_list: List[int],
) -> pd.DataFrame:
    """
    For each buffer in buffer_list:
      - run buffer_intersect(hpm_gdf, census_gdf, buffer)
      - collect the per-buffer 'geoid20' + summed AADT column (e.g., 'aadt_500')
    Finally, merge all ten tables on 'geoid20' and return a single DataFrame.

    Parameters
    ----------
    hpm_gdf : GeoDataFrame
        HPMS roads GeoDataFrame (projected CRS recommended; units in meters).
    census_gdf : GeoDataFrame
        Census blocks GeoDataFrame (same CRS and units as hpm_gdf).
    buffer_list : List[int]
        List of buffer distances (meters).
    join_how : {"outer","inner","left","right"}
        Merge style for combining all per-buffer outputs. Default 'outer' keeps all geoid20s.

    Returns
    -------
    DataFrame
        A single DataFrame with one row per 'geoid20' and one column per buffer (e.g., 'aadt_500', 'aadt_450', ...).
    """
    # Collect per-buffer results as DataFrames indexed by geoid20
    frames = []
    for b in buffer_list:
        # IMPORTANT: use copies so the in-place buffer does not compound across loops
        agg_b = buffer_intersect(hpm_gdf.copy(), census_gdf.copy(), b)
        # Ensure expected columns exist: 'geoid20' and f'aadt_{b}'
        if "geoid20" not in agg_b.columns or f"aadt_{b}" not in agg_b.columns:
            raise KeyError(f"Expected columns 'geoid20' and 'aadt_{b}' missing in result for buffer {b}.")
        frames.append(agg_b.set_index("geoid20"))

    # Fast merge by index: concatenate columns aligned on 'geoid20'
    merged = pd.concat(frames, axis=1).reset_index()

    return merged


In [88]:
final_df = run_buffers_and_merge(hpm_gdf=hpm_df2, census_gdf=census_gdf,buffer_list = buffer_list)

In [89]:
final_df.shape

(122303, 11)

In [90]:
final_df.head(1)

,geoid20,aadt_500,aadt_450,aadt_400,aadt_350,aadt_300,aadt_250,aadt_200,aadt_150,aadt_100,aadt_50
0,060014001001030,316000,316000.0,316000.0,316000.0,158000.0,158000.0,158000.0,158000.0,158000.0,158000.0


In [91]:
final_df.geoid20.nunique()

122303

In [92]:
census_gdf.geoid20.nunique()

519723